#### Create a project

In [ ]:
%%bash
PROJECT=/work/rr151/Julia_Riley

mkdir -p "$PROJECT"/{fastqs,reference,inputs,runs,logs,scripts,results}
cd $PROJECT

##### Add fastqs and references 

In [ ]:
%%bash
# add FASTQs under
cp /path/*fastq.gz /work/rr151/Julia_Riley/fastqs
# add references under
cp /hpc/group/gersbachlab/rr151/reference/ENCODE_RNAseq/hg38/* /work/rr151/Julia_Riley/reference

In [ ]:
%%bash
# Add samples.tsv under PROJECT
PROJECT=/work/rr151/Julia_Riley

cat > "$PROJECT/samples.tsv" <<'EOF'
condition	sample	replicate	R1	R2
ab-ex	D1-ab-ex	D1	D1-ab-ex.R1.fastq.gz	D1-ab-ex.R2.fastq.gz
cntrl	D1-cntrl	D1	D1-cntrl.R1.fastq.gz	D1-cntrl.R2.fastq.gz
dyna-ex	D1-dyna-ex	D1	D1-dyna-ex.R1.fastq.gz	D1-dyna-ex.R2.fastq.gz
ab-ex	D2-ab-ex	D2	D2-ab-ex.R1.fastq.gz	D2-ab-ex.R2.fastq.gz
cntrl	D2-cntrl	D2	D2-cntrl.R1.fastq.gz	D2-cntrl.R2.fastq.gz
dyna-ex	D2-dyna-ex	D2	D2-dyna-ex.R1.fastq.gz	D2-dyna-ex.R2.fastq.gz
ab-ex	D3-ab-ex	D3	D3-ab-ex.R1.fastq.gz	D3-ab-ex.R2.fastq.gz
cntrl	D3-cntrl	D3	D3-cntrl.R1.fastq.gz	D3-cntrl.R2.fastq.gz
dyna-ex	D3-dyna-ex	D3	D3-dyna-ex.R1.fastq.gz	D3-dyna-ex.R2.fastq.gz
EOF

##### Generate input JSON files

In [ ]:

import csv
import json
import os
from pathlib import Path

PROJECT = Path("/work/rr151/Julia_Riley")
FASTQ = PROJECT / "fastqs"
REF = PROJECT / "reference"
INPUTS = PROJECT / "inputs"

INPUTS.mkdir(parents=True, exist_ok=True)

STAR = REF / "GRCh38_V29_UCSC_names_ERCC_phiX_starIndex.tgz"
RSEM = REF / "GRCh38_V29_UCSC_names_ERCC_phiX_rsemIndex.tgz"
KALLISTO = REF / "gencode.v29.transcripts_ERCC_phiX.idx"
CHROM_SIZES = REF / "GRCh38_EBV.chrom.sizes.tsv"
TX_MAPPING = REF / "gencodeV29pri-UCSC-tRNAs-ERCC-phiX.transcript_id_to_genes.tsv"

# check references exist before generating jobs
for f in [STAR, RSEM, KALLISTO, CHROM_SIZES, TX_MAPPING]:
    if not f.exists():
        raise FileNotFoundError(f"Missing reference: {f}")

with open(PROJECT / "samples.tsv") as handle:
    reader = csv.DictReader(handle, delimiter="\t")
    for row in reader:
        sample = row["sample"]
        r1 = FASTQ / row["R1"]
        r2 = FASTQ / row["R2"]
        if not r1.exists():
            raise FileNotFoundError(f"Missing FASTQ: {r1}")
        if not r2.exists():
            raise FileNotFoundError(f"Missing FASTQ: {r2}")
        config = {
            "rna.align_disk": "local-disk 150 HDD",
            "rna.align_index": str(STAR),
            "rna.align_ncpus": 8,
            "rna.align_ramGB": 48,
            "rna.bam_to_signals_disk": "local-disk 100 HDD",
            "rna.bam_to_signals_ncpus": 8,
            "rna.bam_to_signals_ramGB": 48,
            "rna.bamroot": sample,
            "rna.chrom_sizes": str(CHROM_SIZES),
            "rna.endedness": "paired",
            "rna.fastqs_R1": [
                [str(r1)]
            ],
            "rna.fastqs_R2": [
                [str(r2)]
            ],
            "rna.kallisto_disk": "local-disk 50 HDD",
            "rna.kallisto_index": str(KALLISTO),
            "rna.kallisto_number_of_threads": 8,
            "rna.kallisto_ramGB": 48,
            "rna.mad_qc_disk": "local-disk 50 HDD",
            "rna.rna_qc_disk": "local-disk 50 HDD",
            "rna.rna_qc_tr_id_to_gene_type_tsv": str(TX_MAPPING),
            "rna.rsem_disk": "local-disk 100 HDD",
            "rna.rsem_index": str(RSEM),
            "rna.rsem_ncpus": 8,
            "rna.rsem_ramGB": 48,
            "rna.strandedness": "unstranded",
            "rna.strandedness_direction": "unstranded"
        }

        output = INPUTS / f"{sample}.json"
        with open(output, "w") as out:
            json.dump(config, out, indent=2)
        print(f"Created: {output}")

##### run all samples step 1

%%writefile /work/rr151/Julia_Riley/script/run_encode_array.sh
#!/bin/bash
#SBATCH --job-name=encode_rna
#SBATCH --array=1-9%3
#SBATCH --cpus-per-task=8
#SBATCH --mem=64G
#SBATCH --time=24:00:00
#SBATCH --output=/work/rr151/Julia_Riley/logs/slurm-%A_%a.out
#SBATCH --error=/work/rr151/Julia_Riley/logs/slurm-%A_%a.err

set -euo pipefail

PROJECT="/work/rr151/Julia_Riley"

WDL="/hpc/group/gersbachlab/rr151/software/ENCODE_DCC/rna-seq-pipeline/rna-seq-pipeline.wdl"

LINE=$((SLURM_ARRAY_TASK_ID + 1))

IFS=$'\t' read -r CONDITION SAMPLE REPLICATE R1 R2 \
    < <(sed -n "${LINE}p" "$PROJECT/samples.tsv")

if [[ -z "${SAMPLE:-}" ]]; then
    echo "ERROR: No sample found for array task $SLURM_ARRAY_TASK_ID"
    exit 1
fi

echo "========================================"
echo "Sample:       $SAMPLE"
echo "SLURM job:    $SLURM_JOB_ID"
echo "Array task:   $SLURM_ARRAY_TASK_ID"
echo "Node:         $(hostname)"
echo "CPUs:         $SLURM_CPUS_PER_TASK"
echo "Start:        $(date)"
echo "========================================"

INPUT="${PROJECT}/inputs/${SAMPLE}.json"
RUNDIR="${PROJECT}/runs/${SAMPLE}"

if [[ ! -f "$INPUT" ]]; then
    echo "ERROR: Input JSON not found:"
    echo "$INPUT"
    exit 1
fi

mkdir -p "$RUNDIR"

cd "$RUNDIR"

caper run "$WDL" \
    -i "$INPUT" \
    -m metadata.json \
    -b local \
    --singularity

echo "========================================"
echo "Finished: $SAMPLE"
echo "End:      $(date)"
echo "========================================"

##### run all samples step 2

In [ ]:
%%bash
cd /work/rr151/Julia_Riley
sbatch scripts/run_encode_array.sh

In [2]:
%%writefile /work/rr151/Julia_Riley/scripts/collect_outputs.sh
#!/bin/bash

set -euo pipefail

PROJECT="/work/rr151/Julia_Riley"
SAMPLE_SHEET="$PROJECT/samples.tsv"

mkdir -p "$PROJECT/results"

# Read samples.tsv
# Expected columns:
# condition  sample  replicate  R1  R2
while IFS=$'\t' read -r CONDITION SAMPLE REPLICATE R1 R2
do
    # Skip header
    [[ "$SAMPLE" == "sample" ]] && continue

    # Skip empty lines
    [[ -z "$SAMPLE" ]] && continue

    echo "========================================"
    echo "Collecting: $SAMPLE"
    echo "Condition:  $CONDITION"
    echo "Replicate:  $REPLICATE"
    echo "========================================"

    META="$PROJECT/runs/$SAMPLE/metadata.json"
    OUT="$PROJECT/results/$SAMPLE"

    if [[ ! -f "$META" ]]; then
        echo "WARNING: no metadata.json for $SAMPLE"
        echo "Skipping $SAMPLE"
        continue
    fi

    mkdir -p \
        "$OUT/bam" \
        "$OUT/signal" \
        "$OUT/rsem" \
        "$OUT/kallisto" \
        "$OUT/qc"

    # RSEM gene-level quantification
    find "$PROJECT/runs/$SAMPLE" \
        -type f -name "*.genes.results" \
        -exec cp -L {} "$OUT/rsem/" \;

    # RSEM transcript/isoform quantification
    find "$PROJECT/runs/$SAMPLE" \
        -type f -name "*.isoforms.results" \
        -exec cp -L {} "$OUT/rsem/" \;

    # Kallisto abundance
    find "$PROJECT/runs/$SAMPLE" \
        -type f -name "*abundance.tsv" \
        -exec cp -L {} "$OUT/kallisto/" \;

    # BAMs
    find "$PROJECT/runs/$SAMPLE" \
        -type f -name "*.bam" \
        -exec cp -L {} "$OUT/bam/" \;

    # BigWig signal tracks
    find "$PROJECT/runs/$SAMPLE" \
        -type f \( -name "*.bw" -o -name "*.bigWig" \) \
        -exec cp -L {} "$OUT/signal/" \;

    # Save workflow metadata
    cp "$META" "$OUT/metadata.json"

    echo "Finished: $SAMPLE"
    echo

done < "$SAMPLE_SHEET"

echo "========================================"
echo "All available outputs collected."
echo "Results: $PROJECT/results"
echo "========================================"

Overwriting /work/rr151/Julia_Riley/script/collect_outputs.sh


In [ ]:
%%bash
#chmod +x /work/rr151/Julia_Riley/scripts/collect_outputs.sh
cd /work/rr151/Julia_Riley/
sbatch scripts/collect_outputs.sh

In [4]:
from pathlib import Path
import pandas as pd

project = Path("/work/rr151/Julia_Riley")
results = project / "results"
sample_sheet = project / "samples.tsv"

samples_df = pd.read_csv(sample_sheet, sep="\t")
required_columns = {"condition", "sample", "replicate", "R1", "R2"}
missing = required_columns - set(samples_df.columns)

if missing:
    raise ValueError(
        f"samples.tsv is missing required columns: {sorted(missing)}"
    )
samples = samples_df["sample"].tolist()
print(f"Found {len(samples)} samples:")
for sample in samples:
    print(f"  {sample}")

# Read RSEM results

counts = []
tpms = []

for sample in samples:
    files = list(
        (results / sample / "rsem").glob("*.genes.results")
    )
    if len(files) != 1:
        raise RuntimeError(
            f"{sample}: expected exactly one genes.results file, "
            f"found {len(files)}"
        )
    print(f"Reading {sample}: {files[0].name}")
    df = pd.read_csv(files[0], sep="\t")
    # Expected counts
    count = df[
        ["gene_id", "expected_count"]
    ].copy()
    count = count.rename(
        columns={"expected_count": sample}
    )
    counts.append(count)
    # TPM
    tpm = df[
        ["gene_id", "TPM"]
    ].copy()
    tpm = tpm.rename(
        columns={"TPM": sample}
    )
    tpms.append(tpm)

# Combine expected counts

count_matrix = counts[0]
for df in counts[1:]:

    count_matrix = count_matrix.merge(
        df,
        on="gene_id",
        how="outer"
    )

# Combine TPM

tpm_matrix = tpms[0]
for df in tpms[1:]:
    tpm_matrix = tpm_matrix.merge(
        df,
        on="gene_id",
        how="outer"
    )

# Save matrices

count_output = results / "RSEM_expected_counts.tsv"
tpm_output = results / "RSEM_TPM.tsv"

count_matrix.to_csv(
    count_output,
    sep="\t",
    index=False
)
tpm_matrix.to_csv(
    tpm_output,
    sep="\t",
    index=False
)

print("Created:")
print(results / "RSEM_expected_counts.tsv")
print(results / "RSEM_TPM.tsv")


Created:
/work/rr151/Julia_Riley/results/RSEM_expected_counts.tsv
/work/rr151/Julia_Riley/results/RSEM_TPM.tsv
